# 05 — MLOps: From Theory to Automated Pipelines

## Part 1: Theory

### 1.1 What is MLOps?

MLOps = **DevOps principles applied to ML systems**. Bridges the gap between model development and production.

```
Traditional Software:  Code → Build → Test → Deploy → Monitor
ML Systems:           Data → Train → Evaluate → Deploy → Monitor → Retrain (loop)
```

### 1.2 MLOps Maturity Levels

| Level | Description | Characteristics |
|-------|-------------|----------------|
| 0 | Manual | Jupyter notebooks, manual deployment, no monitoring |
| 1 | ML Pipeline | Automated training, basic CI/CD, manual triggers |
| 2 | CI/CD | Automated testing + deployment, feature store |
| 3 | Full Auto | Automated retraining on drift, A/B testing, canary |
| 4 | Self-healing | Auto-rollback, auto-scaling, self-optimizing |

### 1.3 Three Pillars of Versioning

```
┌─────────────────────────────────────────────────┐
│              Reproducibility                      │
│  ┌──────────┐  ┌──────────┐  ┌──────────────┐  │
│  │   Code   │  │   Data   │  │    Model     │  │
│  │  (Git)   │  │  (DVC)   │  │  (MLflow)    │  │
│  └──────────┘  └──────────┘  └──────────────┘  │
└─────────────────────────────────────────────────┘
```

### 1.4 Types of Drift

| Type | What Changes | Detection | Example |
|------|-------------|-----------|----------|
| **Data drift** | Input distribution P(X) | KS test, PSI | Users start asking different questions |
| **Concept drift** | Relationship P(Y|X) | Performance monitoring | Correct answers change over time |
| **Model drift** | Model performance | Accuracy metrics | Model degrades on new patterns |

**Statistical Tests:**
- **KS Test**: max|F₁(x) - F₂(x)| — tests if two distributions differ
- **PSI**: Σ(P_i - Q_i) × ln(P_i/Q_i) — population stability index
- **Wasserstein**: Earth mover's distance between distributions

### 1.5 LLMOps (MLOps for LLMs)

| Traditional MLOps | LLMOps |
|-------------------|--------|
| Train models | Prompt engineering + RAG |
| Feature engineering | Chunking + embedding |
| Model versioning | Prompt versioning |
| Data drift | Query drift + retrieval quality |
| A/B test predictions | A/B test generations |
| Retrain on new data | Update knowledge base |

---

## Part 2: Implementation

In [ ]:
import mlflow
import numpy as np
import pandas as pd
import json
from datetime import datetime, timedelta
from scipy import stats

### MLflow Experiment Tracking

In [ ]:
mlflow.set_experiment("rag-pipeline-experiments")

experiments = [
    {"config": {"run_name": "baseline", "chunk_size": 500, "overlap": 50, "top_k": 3, "model": "llama-3.1-8b"},
     "metrics": {"retrieval_recall": 0.72, "answer_similarity": 0.81, "latency_p50_ms": 850, "cost_per_query": 0.0}},
    {"config": {"run_name": "larger_chunks", "chunk_size": 1000, "overlap": 100, "top_k": 3, "model": "llama-3.1-8b"},
     "metrics": {"retrieval_recall": 0.68, "answer_similarity": 0.84, "latency_p50_ms": 920, "cost_per_query": 0.0}},
    {"config": {"run_name": "more_context", "chunk_size": 500, "overlap": 50, "top_k": 5, "model": "llama-3.1-8b"},
     "metrics": {"retrieval_recall": 0.85, "answer_similarity": 0.86, "latency_p50_ms": 1100, "cost_per_query": 0.0}},
    {"config": {"run_name": "gemma_model", "chunk_size": 500, "overlap": 50, "top_k": 3, "model": "gemma-2-9b"},
     "metrics": {"retrieval_recall": 0.72, "answer_similarity": 0.79, "latency_p50_ms": 750, "cost_per_query": 0.0}},
]

for exp in experiments:
    with mlflow.start_run(run_name=exp["config"]["run_name"]):
        mlflow.log_params(exp["config"])
        mlflow.log_metrics(exp["metrics"])
        mlflow.set_tag("timestamp", datetime.now().isoformat())
    print(f"Logged: {exp['config']['run_name']} → similarity={exp['metrics']['answer_similarity']}")

print("\nView experiments: mlflow ui")

### Drift Detection (from scratch)

In [ ]:
def ks_test(reference, current):
    """Kolmogorov-Smirnov test for distribution shift."""
    stat, p_value = stats.ks_2samp(reference, current)
    return {"statistic": stat, "p_value": p_value, "drift_detected": p_value < 0.05}

def psi(reference, current, bins=10):
    """Population Stability Index."""
    ref_hist, edges = np.histogram(reference, bins=bins, density=True)
    cur_hist, _ = np.histogram(current, bins=edges, density=True)
    # Avoid division by zero
    ref_hist = np.clip(ref_hist, 1e-6, None)
    cur_hist = np.clip(cur_hist, 1e-6, None)
    # Normalize
    ref_pct = ref_hist / ref_hist.sum()
    cur_pct = cur_hist / cur_hist.sum()
    psi_value = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return {"psi": psi_value, "drift_detected": psi_value > 0.2}  # >0.2 = significant shift

# Simulate reference (training) vs current (production) distributions
np.random.seed(42)
ref_query_lengths = np.random.normal(50, 10, 1000)
ref_retrieval_scores = np.random.normal(0.82, 0.08, 1000)

# Current data WITH drift
cur_query_lengths = np.random.normal(65, 12, 1000)  # queries getting longer!
cur_retrieval_scores = np.random.normal(0.72, 0.12, 1000)  # quality dropping!

print("=== Drift Detection Results ===")
for name, ref, cur in [("query_length", ref_query_lengths, cur_query_lengths),
                        ("retrieval_score", ref_retrieval_scores, cur_retrieval_scores)]:
    ks = ks_test(ref, cur)
    ps = psi(ref, cur)
    print(f"\n{name}:")
    print(f"  KS test: stat={ks['statistic']:.4f}, p={ks['p_value']:.6f}, drift={ks['drift_detected']}")
    print(f"  PSI: {ps['psi']:.4f}, drift={ps['drift_detected']}")

### Automated Retraining Trigger

In [ ]:
class DriftMonitor:
    def __init__(self, threshold_psi=0.2, threshold_ks_p=0.05):
        self.threshold_psi = threshold_psi
        self.threshold_ks_p = threshold_ks_p
        self.alerts = []
    
    def check(self, reference, current, metric_name):
        ks = ks_test(reference, current)
        ps = psi(reference, current)
        drifted = ks["drift_detected"] or ps["drift_detected"]
        if drifted:
            self.alerts.append({"metric": metric_name, "ks_p": ks["p_value"], "psi": ps["psi"], "time": datetime.now().isoformat()})
        return drifted
    
    def should_retrain(self, min_alerts=2):
        if len(self.alerts) >= min_alerts:
            print(f"⚠️  RETRAIN TRIGGERED: {len(self.alerts)} metrics drifted")
            for a in self.alerts:
                print(f"   - {a['metric']}: PSI={a['psi']:.4f}")
            return True
        print(f"✅ OK: {len(self.alerts)}/{min_alerts} drift alerts")
        return False

monitor = DriftMonitor()
monitor.check(ref_query_lengths, cur_query_lengths, "query_length")
monitor.check(ref_retrieval_scores, cur_retrieval_scores, "retrieval_score")
monitor.should_retrain()

### CI/CD Pipeline (GitHub Actions)

In [ ]:
cicd_yaml = """
# .github/workflows/rag-pipeline.yml
name: RAG Pipeline CI/CD

on:
  push:
    paths: ['data/**', 'scripts/**', 'prompts/**']
  schedule:
    - cron: '0 6 * * 1'  # Weekly drift check

jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - run: pip install -r requirements.txt
      - run: python -m pytest tests/ -v

  check-drift:
    needs: validate
    runs-on: ubuntu-latest
    steps:
      - run: python scripts/check_drift.py
      - run: |
          if [ -f drift_detected.flag ]; then
            echo "DRIFT=true" >> $GITHUB_ENV
          fi

  rebuild-index:
    needs: check-drift
    if: env.DRIFT == 'true' || github.event_name == 'push'
    runs-on: ubuntu-latest
    steps:
      - run: python scripts/ingest.py
      - run: python scripts/embed.py
      - run: python scripts/build_index.py

  evaluate:
    needs: rebuild-index
    runs-on: ubuntu-latest
    steps:
      - run: python scripts/evaluate.py
      - run: |
          SCORE=$(cat metrics/similarity.txt)
          if (( $(echo "$SCORE < 0.75" | bc -l) )); then
            echo "Quality gate FAILED: $SCORE < 0.75"
            exit 1
          fi

  deploy:
    needs: evaluate
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - run: sam build && sam deploy --no-confirm-changeset
"""
print(cicd_yaml)

## Part 3: Key Takeaways

1. **Track everything**: params, metrics, artifacts — reproducibility is non-negotiable
2. **Version data alongside code** — DVC or similar
3. **Detect drift early** — KS test + PSI on key distributions
4. **Automate the loop** — drift → retrain → evaluate → deploy
5. **Quality gates** prevent bad models from reaching production
6. **LLMOps** differs from traditional MLOps — prompt versioning, retrieval quality monitoring

### Next → 06_cloud_deployment.ipynb